In [20]:
import wrds
import pandas as pd
import numpy as np

In [14]:
data_location = r"C:\Github Code\quant-factor-engine\data"

# 1. Connect
db = wrds.Connection(wrds_username='YOUR_USERNAME')

# ==========================================
# QUERY 1: THE FUNDAMENTALS (Compustat)
# ==========================================
# We join with the Linktable here so we get the 'PERMNO' (CRSP ID) immediately.
# This is the "Rosetta Stone" that lets you merge this with price data later.

fund_query = """
SELECT 
    a.gvkey, a.datadate, a.tic, a.conm,
    b.lpermno as permno, b.linktype, b.linkenddt,
    -- VALUE
    a.seq, a.ceq, a.pstk, a.txditc,
    -- PROFITABILITY
    a.revt, a.cogs, a.xsga, a.xint, a.ib, a.dp,
    -- GROWTH / INVESTMENT
    a.at, a.capx, a.ppent,
    -- ACCRUALS / LEVERAGE
    a.act, a.lct, a.che, a.dltt, a.dlc, a.invt, a.rect
FROM 
    comp.funda a
LEFT JOIN 
    crsp.ccmxpf_linktable b 
    ON a.gvkey = b.gvkey
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C') 
WHERE 
    a.datadate >= '01/01/2010'
    AND a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
"""
print("Fetching Fundamentals...")
df_fund = db.raw_sql(fund_query)
# Save immediately
df_fund.to_parquet(data_location + r'\compustat_fundamentals.parquet', engine='fastparquet')


# ==========================================
# QUERY 2: MONTHLY STOCK FILE (CRSP MSF)
# ==========================================
# Use this for: Momentum, Market Cap, Beta, and Monthly Returns
# We filter by share code 10 or 11 to get only "Common Stocks" (no ETFs/ADRs in factor construction usually)

msf_query = """
SELECT 
    a.permno, a.date, 
    a.prc, a.ret, a.shrout, a.vol, a.cfacpr, a.cfacshr
FROM 
    crsp.msf a
LEFT JOIN 
    crsp.msenames b 
    ON a.permno = b.permno
    AND b.namedt <= a.date 
    AND a.date <= b.nameendt
WHERE 
    a.date >= '01/01/2010'
    AND b.shrcd IN (10, 11) -- Only US Common Stocks
"""
print("Fetching Monthly Prices...")
df_msf = db.raw_sql(msf_query)
df_msf.to_parquet(data_location + r'\crsp_monthly.parquet', engine='fastparquet')


# ==========================================
# QUERY 3: DAILY STOCK FILE (CRSP DSF)
# ==========================================
# Use this for: GARCH, Volatility, VaR
# Warning: This file is HUGE. We only grab 'ret' (returns) to save space.

dsf_query = """
SELECT 
    permno, date, ret
FROM 
    crsp.dsf
WHERE 
    date >= '01/01/2010'
"""
print("Fetching Daily Prices...")
df_dsf = db.raw_sql(dsf_query)
df_dsf.to_parquet(data_location + r'\crsp_daily.parquet', engine='fastparquet')

print("All downloads complete.")
db.close()

Enter your WRDS username [Admin]: xenith
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\Admin\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Fetching Fundamentals...
Fetching Monthly Prices...
Fetching Daily Prices...
All downloads complete.


In [29]:
# 1. Load Data
print("Loading raw data...")
df_fund = pd.read_parquet(data_location + r'\compustat_fundamentals.parquet', engine='fastparquet')
df_crsp = pd.read_parquet(data_location + r'\crsp_monthly.parquet', engine='fastparquet')

# ==========================================
# STEP 1: PREP FUNDAMENTALS (COMPUSTAT)
# ==========================================
print("Calculating fundamental factors...")

# Handle missing values (fill with 0 for calculation safety, or drop)
df_fund = df_fund.fillna(0)

# A. Value Factor: Book Value of Equity
# BE = Stockholders' Equity + Deferred Taxes - Preferred Stock
df_fund['be'] = df_fund['seq'] + df_fund['txditc'] - df_fund['pstk']
# Avoid negative book value (doesn't make sense for factors)
df_fund = df_fund[df_fund['be'] > 0]

# B. Quality Factor: Operating Profitability
# OP = (Revenue - COGS - SG&A) / Book Equity
df_fund['op_prof'] = (df_fund['revt'] - df_fund['cogs'] - df_fund['xsga']) / df_fund['be']

# C. Investment Factor: Asset Growth
# Growth = (Assets_t / Assets_t-1) - 1
df_fund = df_fund.sort_values(['gvkey', 'datadate'])
df_fund['asset_growth'] = df_fund.groupby('gvkey')['at'].pct_change()

# D. Setup "Valid From" Date (Crucial for Look-Ahead Bias)
# We assume data is public 6 months after fiscal year end
df_fund['datadate'] = pd.to_datetime(df_fund['datadate'])
df_fund['public_date'] = df_fund['datadate'] + pd.DateOffset(months=6)

# Keep only what we need
fund_cols = ['permno', 'public_date', 'be', 'op_prof', 'asset_growth']
df_fund = df_fund[fund_cols].dropna()


# ==========================================
# STEP 2: PREP PRICES (CRSP MONTHLY)
# ==========================================
print("Calculating price-based factors...")

# Fix Dates
df_crsp['date'] = pd.to_datetime(df_crsp['date'])
df_crsp = df_crsp.sort_values(['permno', 'date'])

# A. Market Cap (Price * Shares Outstanding)
# CRSP prices are negative if they are averages of bid/ask. Take abs().
df_crsp['mkt_cap'] = df_crsp['prc'].abs() * df_crsp['shrout']

# B. Momentum (12-1 Month Return)
# We log-transform returns for summation: r_log = ln(1+r)
df_crsp['log_ret'] = np.log(1 + df_crsp['ret'])
# Rolling 11 month sum, shifted by 2 months (to skip the most recent month t-1)
# Actually standard definition: Sum of t-12 to t-2 returns.
# We skip t-1 to avoid "Short Term Reversal" microstructure noise.
df_crsp['momentum'] = df_crsp.groupby('permno')['log_ret'].transform(
    lambda x: x.rolling(window=11).sum().shift(2)
)

# ==========================================
# STEP 3: MERGE (The "As-Of" Join)
# ==========================================
print("Merging datasets...")

df_crsp["permno"] = df_crsp["permno"].astype("float64")

df_crsp = df_crsp.sort_values('date')
df_fund = df_fund.sort_values('public_date')

# merge_asof matches the CRSP 'date' with the CLOSEST PREVIOUS 'public_date'
# This ensures we only use data that was publicly available at the time of trading.
df_combined = pd.merge_asof(
    df_crsp, 
    df_fund, 
    left_on='date', 
    right_on='public_date', 
    by='permno',
    direction='backward' # Only look in the past!
)

# ==========================================
# STEP 4: FINAL CLEANUP & SAVE
# ==========================================
# Calculate Book-to-Market (Value Factor) using current Market Cap
df_combined['b_m'] = df_combined['be'] * 1000 / df_combined['mkt_cap'] 
# Note: Compustat 'seq' is in millions, CRSP 'mkt_cap' is in thousands usually. 
# Check units! Usually WRDS Compustat is Millions, CRSP is thousands. 
# If so, multiply BE by 1000. *VERIFY THIS IN DATA INSPECTION*

# Filter: Remove tiny stocks (Penny stocks break quant models)
df_final = df_combined[df_combined['mkt_cap'] > 100000].copy() # >$100M Cap

print(f"Final Universe Shape: {df_final.shape}")
df_final.to_parquet(data_location + r'\factor_data.parquet', engine='fastparquet')
print("Saved to data/factor_data.parquet")

Loading raw data...
Calculating fundamental factors...
Calculating price-based factors...
Merging datasets...
Final Universe Shape: (535924, 16)
Saved to data/factor_data.parquet


In [ ]:
df_fund